In [45]:
# Data path for R library 

# /var/folders/lt/fs9cxhhn7_v8qvhkq56m6zh80000gn/T//RtmpQEZlmN/downloaded_packages

In [46]:
!playwright install chromium

In [47]:
import requests
from bs4 import BeautifulSoup
import json

# Use the standard readable URL
url = "https://www.fotmob.com/players/422685/bruno-fernandes"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Next.js stores all the page data in this specific tag
    next_data_script = soup.find('script', id='__NEXT_DATA__')
    
    if next_data_script:
        # Load the text inside the script tag as a python dictionary
        json_data = json.loads(next_data_script.string)
        
        # The data you need is nested inside the props
        # Let's print the keys to help you navigate it
        print("Data extracted successfully. Top level keys:")
        print(json_data.keys())
        
        # To get you started, the core stats are usually under:
        # page_props = json_data['props']['pageProps']
    else:
        print("Could not find the __NEXT_DATA__ script tag.")
else:
    print(f"Request failed. Status code: {response.status_code}")

Data extracted successfully. Top level keys:
dict_keys(['props', 'page', 'query', 'buildId', 'isFallback', 'isExperimentalCompile', 'dynamicIds', 'gssp', 'appGip', 'scriptLoader'])


In [48]:
# Dig into the 'props' key
page_props = json_data['props']['pageProps']

print("Keys inside pageProps:")
print(page_props.keys())

# In many Next.js sites, the actual API response is cached inside a 'fallback' dictionary
if 'fallback' in page_props:
    fallback_data = page_props['fallback']
    
    # The keys in 'fallback' are usually the API URLs themselves
    # Let's get the first key and look inside
    api_url_key = list(fallback_data.keys())[0]
    player_data = fallback_data[api_url_key]
    
    print("\nSuccess! Here are the keys for the actual player data:")
    if isinstance(player_data, dict):
        print(player_data.keys())
else:
    # If no fallback, it might be directly in pageProps
    print("\nNo fallback found, check pageProps directly for stats/matches.")

Keys inside pageProps:
dict_keys(['data', 'fallback', 'translations'])

Success! Here are the keys for the actual player data:
dict_keys(['id', 'name', 'birthDate', 'contractEnd', 'isCoach', 'isCaptain', 'gender', 'primaryTeam', 'positionDescription', 'injuryInformation', 'internationalDuty', 'playerInformation', 'mainLeague', 'trophies', 'recentMatches', 'careerHistory', 'traits', 'meta', 'coachStats', 'statSeasons', 'firstSeasonStats', 'status', 'marketValues', 'relatedLinksData', 'nextMatch', 'dataProvider', 'ssr'])


In [49]:
# Extract the list of recent matches
recent_matches = player_data.get('recentMatches', [])

if recent_matches:
    print(f"Found {len(recent_matches)} recent matches!")
    
    # Grab the very first match in the list to inspect its structure
    first_match = recent_matches[0]
    
    print("\nKeys inside a single match object:")
    print(first_match.keys())
    
    # Let's print out some of the juicy details if they exist
    print("\nSample Match Data:")
    print(f"Match ID: {first_match.get('id', 'Not found')}")
    print(f"Opponent: {first_match.get('opponent', {}).get('name', 'Not found')}")
    print(f"Player Rating: {first_match.get('rating', 'Not found')}")
    
else:
    print("No recent matches found in this dataset.")

Found 56 recent matches!

Keys inside a single match object:
dict_keys(['teamId', 'teamName', 'opponentTeamId', 'opponentTeamName', 'isHomeTeam', 'id', 'matchDate', 'matchPageUrl', 'leagueId', 'leagueName', 'stage', 'homeScore', 'awayScore', 'minutesPlayed', 'goals', 'assists', 'yellowCards', 'redCards', 'ratingProps', 'playerOfTheMatch', 'onBench'])

Sample Match Data:
Match ID: 4813697
Opponent: Not found
Player Rating: Not found


In [50]:
import requests
from bs4 import BeautifulSoup
import json

match_id = "4813697"

# Use the standard front-facing web URL instead of the dead API endpoint
url = f"https://www.fotmob.com/match/{match_id}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    next_data_script = soup.find('script', id='__NEXT_DATA__')
    
    if next_data_script:
        json_data = json.loads(next_data_script.string)
        page_props = json_data['props']['pageProps']
        
        print("Success, bypassed the 404. Keys inside the Next.js Match pageProps:")
        print(page_props.keys())
        
        # Digging into the fallback cache where the match stats usually live
        if 'fallback' in page_props:
            fallback_data = page_props['fallback']
            
            # Grab the first key in the fallback dictionary
            api_url_key = list(fallback_data.keys())[0]
            match_details = fallback_data[api_url_key]
            
            print("\nKeys inside the actual match details:")
            if isinstance(match_details, dict):
                print(match_details.keys())
                
                # Check if the specific lineup data is nested in here
                if 'content' in match_details and 'lineup' in match_details['content']:
                    print("\nFound the lineup data! Keys inside lineup:")
                    print(match_details['content']['lineup'].keys())
    else:
        print("Could not find the __NEXT_DATA__ script tag.")
else:
    print(f"Request failed. Status code: {response.status_code}")

Success, bypassed the 404. Keys inside the Next.js Match pageProps:
dict_keys(['general', 'header', 'nav', 'ongoing', 'hasPendingVAR', 'content', 'seo', 'fetchingLeagueData', 'ssr', 'fallback', 'translations'])

Keys inside the actual match details:
dict_keys(['matches'])


In [51]:
import pandas as pd

# Using your exact file path and fixing the Dtype warning
df = pd.read_csv('/Users/schoudhry/Desktop/mind-overnight/code/epl_match_details.csv', low_memory=False)

# Print all the column names so we can see the exact labels
print("Available columns in the dataset:")
columns = df.columns.tolist()

# Grouping them by 5 so it is easy to read
for i in range(0, len(columns), 5):
    print(columns[i:i+5])

Available columns in the dataset:
['match_id', 'match_round', 'league_id', 'league_name', 'league_round_name']
['parent_league_id', 'parent_league_season', 'match_time_utc', 'home_team_id', 'home_team']
['home_team_color', 'away_team_id', 'away_team', 'away_team_color', 'id']
['event_type', 'team_id', 'player_id', 'player_name', 'x']
['y', 'min', 'min_added', 'is_blocked', 'is_on_target']
['goal_crossed_y', 'expected_goals', 'expected_goals_on_target', 'shot_type', 'situation']
['period', 'is_own_goal', 'on_goal_shot_x', 'on_goal_shot_y', 'on_goal_shot_zoom_ratio']
['first_name', 'last_name', 'team_color', 'short_name', 'blocked_x']
['blocked_y', 'goal_crossed_z', 'full_name', 'is_saved_off_line', 'is_from_inside_box']


In [52]:
!playwright install chromium

In [53]:
!pip install sofascore-wrapper

In [54]:
import asyncio
from sofascore_wrapper.api import SofascoreAPI
from sofascore_wrapper.search import Search

async def main():
    api = SofascoreAPI()
    
    # Search for the player
    search = Search(api, search_string="saka")
    raw_data = await search.search_all()
    
    # Loop through the massive results list
    for item in raw_data.get('results', []):
        
        # We only care if the result is actually a player
        if item.get('type') == 'player':
            entity = item['entity']
            player_id = entity['id']
            name = entity['name']
            team = entity['team']['name']
            
            print("Match Found:")
            print(f"Name: {name}")
            print(f"Player ID: {player_id}")
            print(f"Current Team: {team}")
            
            # Stop the loop after finding the first most relevant player
            break

    await api.close()

if __name__ == "__main__":
    asyncio.run(main())
    
    

Match Found:
Name: Bukayo Saka
Player ID: 934235
Current Team: Arsenal


In [55]:
import requests
import pandas as pd
import time

player_id = 934235 # Bukayo Saka

# 1. Get his recent matches
matches_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/0"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Origin": "https://www.sofascore.com",
    "Referer": "https://www.sofascore.com/"
}

response = requests.get(matches_url, headers=headers)
matches_data = response.json().get('events', [])

print(f"Found {len(matches_data)} recent matches. Extracting ratings...")

player_ratings = []

# 2. Loop through the matches to get his specific rating
# Limiting to 5 for the test run so we don't hit rate limits
for event in matches_data[:5]:
    match_id = event['id']
    home_team = event['homeTeam']['name']
    away_team = event['awayTeam']['name']
    matchup = f"{home_team} vs {away_team}"
    
    # Hit the lineups endpoint for this specific match
    lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
    lineup_resp = requests.get(lineup_url, headers=headers)
    
    rating = None
    if lineup_resp.status_code == 200:
        lineup_data = lineup_resp.json()
        
        # Search both home and away teams for our player
        for team_key in ['home', 'away']:
            if team_key in lineup_data:
                for player_node in lineup_data[team_key].get('players', []):
                    if player_node.get('player', {}).get('id') == player_id:
                        # Extract the rating
                        rating = player_node.get('statistics', {}).get('rating')
                        break
            if rating:
                break
                
    player_ratings.append({
        'Match_ID': match_id,
        'Matchup': matchup,
        'Rating': rating
    })
        
    # Be polite to the API to avoid 403 blocks
    time.sleep(0.5)

df_ratings = pd.DataFrame(player_ratings)
print("\nSaka's Match Ratings:")
print(df_ratings)

Found 0 recent matches. Extracting ratings...

Saka's Match Ratings:
Empty DataFrame
Columns: []
Index: []


In [56]:
!pip install cloudscraper

In [57]:
import cloudscraper
import pandas as pd
import time

player_id = 934235 # Bukayo Saka

# cloudscraper automatically handles the headers and bot bypass
scraper = cloudscraper.create_scraper()

matches_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/0"

response = scraper.get(matches_url)
matches_data = response.json().get('events', [])

print(f"Found {len(matches_data)} recent matches. Extracting ratings...")

player_ratings = []

# Limiting to 5 for the test run
for event in matches_data[:5]:
    match_id = event['id']
    home_team = event['homeTeam']['name']
    away_team = event['awayTeam']['name']
    matchup = f"{home_team} vs {away_team}"
    
    lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
    lineup_resp = scraper.get(lineup_url)
    
    rating = None
    if lineup_resp.status_code == 200:
        lineup_data = lineup_resp.json()
        
        for team_key in ['home', 'away']:
            if team_key in lineup_data:
                for player_node in lineup_data[team_key].get('players', []):
                    if player_node.get('player', {}).get('id') == player_id:
                        rating = player_node.get('statistics', {}).get('rating')
                        break
            if rating:
                break
                
    player_ratings.append({
        'Match_ID': match_id,
        'Matchup': matchup,
        'Rating': rating
    })
        
    time.sleep(0.5)

df_ratings = pd.DataFrame(player_ratings)
print("\nSaka's Match Ratings:")
print(df_ratings)

ModuleNotFoundError: No module named 'cloudscraper'